In [2]:
# Data Cleaing:
import os
import pandas as pd

INPUT_PATH = "./Data/processed_transit_data.csv"
OUTPUT_PATH = "./Data/cleaned_transit_data.csv"

if not os.path.exists(INPUT_PATH):
    print(f"Error: {INPUT_PATH} not found.")
    exit()

print("Loading raw processed transit dataset...")
df = pd.read_csv(INPUT_PATH)
initial_count = len(df)
print(f"Initial record count: {initial_count:,}")

# 1. Parse Timestamps
df['departure_timestamp'] = pd.to_datetime(df['departure_timestamp'], errors='coerce')
df = df.dropna(subset=['departure_timestamp'])

# 2. Deduplicate repeated snapshot records for the same trip at the same stop
df = df.sort_values(by=['trip_id', 'stop_id', 'departure_timestamp'])
df = df.drop_duplicates(subset=['trip_id', 'stop_id'], keep='last')
print(f"After deduplication: {len(df):,} rows (removed {initial_count - len(df):,} duplicates)")

# 3. Handle Extreme Outliers
# Removing extreme noise (>2 hours delay or >15 mins early)
df = df[(df['delay_minutes'] >= -15) & (df['delay_minutes'] <= 120)]

# 4. Filter invalid coordinates
df = df.dropna(subset=['stop_lat', 'stop_lon'])
df = df[(df['stop_lat'] != 0) & (df['stop_lon'] != 0)]

# 5. Add Derived Features
df['hour'] = df['departure_timestamp'].dt.hour
df['day_name'] = df['departure_timestamp'].dt.day_name()
df['is_weekend'] = df['departure_timestamp'].dt.dayofweek >= 5

# Map GTFS route_type to vehicle classification
vehicle_map = {0: "Tram", 3: "Bus"}
df['vehicle_type'] = df['route_type'].map(vehicle_map).fillna("Bus")

# 6. Export Cleaned Dataset
df.to_csv(OUTPUT_PATH, index=False)
print(f"\nData cleaning complete!")
print(f"Final clean dataset: {len(df):,} records saved to '{OUTPUT_PATH}'.")

Loading raw processed transit dataset...
Initial record count: 2,647,289
After deduplication: 46,622 rows (removed 2,600,667 duplicates)

Data cleaning complete!
Final clean dataset: 46,576 records saved to './Data/cleaned_transit_data.csv'.
